# ML Coding — Day 3: Attention III (efficient & long-context)

**~1 hour, vectorized NumPy.** Today is about making attention **scale**: streaming/online softmax,
FlashAttention-style tiling, KV-caching, and local windows. **Looping over blocks or timesteps is the
algorithm here and is allowed** — what's banned is per-element Python loops; vectorize *within* each
block. Run the **helpers cell first** (`_softmax`).

**Q1** online softmax stats `[warmup]` · **Q2** FlashAttention forward (tiled) `[medium]` ·
**Q3** KV-cache incremental decode `[medium]` · **Q4** causal FlashAttention (block-sparse) `[hard]` ·
**Q5** sliding-window attention via `sliding_window_view` `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np


def _softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

## Q1 — Online (streaming) softmax statistics  ·  `[warmup]`

**Where it's real:** the numerical core of **FlashAttention** (Dao et al. 2022, arXiv:2205.14135) and
of any softmax over data too large to hold at once. You stream the inputs in blocks while keeping a
running max `m` and running denominator `l`, rescaling `l` whenever the max grows — so you never
materialize all logits and never overflow.

Implement `online_softmax_stats(x, block_size) -> (m, l)` for 1-D `x`: process `x` in consecutive
blocks, returning `m = max(x)` and `l = Σ exp(x − m)` computed via the streaming update
`m_new = max(m, max(block))`, `l = l·exp(m − m_new) + Σ exp(block − m_new)`. Then
`softmax(x) = exp(x − m) / l`. (Block loop OK; vectorize within the block.)

In [ ]:
import numpy as np


def online_softmax_stats(x, block_size):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.standard_normal(50) * 3.0
    m, l = online_softmax_stats(x, 7)
    assert np.isclose(m, x.max())
    assert np.isclose(l, np.exp(x - x.max()).sum())
    p = np.exp(x - m) / l
    assert np.allclose(p, _softmax(x)) and np.isclose(p.sum(), 1.0)
    big = np.array([1000.0, 1000.5, 999.0])          # would overflow naively
    mb, lb = online_softmax_stats(big, 2)
    assert np.isfinite(lb) and np.isclose(mb, 1000.5)
    print("Q1 OK")

_q1()

## Q2 — FlashAttention forward (tiled)  ·  `[medium]`

**Paper:** Dao et al., *FlashAttention*, 2022 (arXiv:2205.14135). **Why it matters:** it computes
exact attention in **O(Tk) memory** (never forming the `Tq×Tk` score matrix) by tiling over key/value
blocks and carrying the online-softmax stats per query — the reason long-context attention is feasible.

Single head, no batch: `Q (Tq,d)`, `K (Tk,d)`, `V (Tk,dv)`. Implement
`flash_attention(Q, K, V, block_size) -> O (Tq,dv)`, equal to standard `softmax(QKᵀ/√d)·V`, by looping
over key blocks and maintaining per-query running `m`, `l`, and rescaled output accumulator. (Block
loop OK; the per-block math is vectorized.)

In [ ]:
def flash_attention(Q, K, V, block_size):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    Tq, Tk, d, dv = 6, 10, 5, 4
    Q = rng.standard_normal((Tq, d)); K = rng.standard_normal((Tk, d)); V = rng.standard_normal((Tk, dv))
    ref = _softmax(Q @ K.T / np.sqrt(d)) @ V
    for bs in (3, 4, Tk):                              # incl. block size that doesn't divide Tk
        assert np.allclose(flash_attention(Q, K, V, bs), ref, atol=1e-10), bs
    print("Q2 OK")

_q2()

## Q3 — KV-cache incremental decode  ·  `[medium]`

**Where it's real:** every autoregressive LLM inference server. During generation you keep the past
keys/values in a cache and, for each new token, **append its k/v and attend over the whole cache** —
making each step O(t) instead of recomputing O(t²).

Implement one decode step `attend_step(q_t, k_t, v_t, K_cache, V_cache) -> (out_t, K_new, V_new)`:
append `k_t (d,)`, `v_t (dv,)` to the caches (`K_cache (t,d)`, `V_cache (t,dv)`), then attend `q_t (d,)`
over the updated cache (`softmax(K·q_t/√d)·V`). Running it step-by-step over a sequence must equal full
**causal** attention.

In [ ]:
def attend_step(q_t, k_t, v_t, K_cache, V_cache):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    T, d, dv = 5, 4, 6
    Qs = rng.standard_normal((T, d)); Ks = rng.standard_normal((T, d)); Vs = rng.standard_normal((T, dv))
    Kc, Vc, outs = np.zeros((0, d)), np.zeros((0, dv)), []
    for t in range(T):                                # generation is inherently sequential
        o, Kc, Vc = attend_step(Qs[t], Ks[t], Vs[t], Kc, Vc)
        outs.append(o)
    inc = np.stack(outs)
    S = np.where(np.tril(np.ones((T, T), bool)), Qs @ Ks.T / np.sqrt(d), -np.inf)
    full = _softmax(S) @ Vs
    assert np.allclose(inc, full, atol=1e-10)
    assert Kc.shape == (T, d) and Vc.shape == (T, dv)
    print("Q3 OK")

_q3()

## Q4 — Causal FlashAttention with block-sparsity  ·  `[hard]`

**Paper:** Dao et al. 2022. **Why it matters:** with a causal mask, whole key blocks that lie entirely
in a query block's future can be **skipped** — the block-sparse pattern that makes causal long-context
attention cheap. The diagonal block needs an in-block causal mask; the trap is producing NaNs from
fully-masked rows.

Self-attention (`Q, K (T,d)`, `V (T,dv)`). Implement `flash_attention_causal(Q, K, V, block_size) -> O`
equal to causal `softmax`-attention, by tiling over query blocks × key blocks, **breaking out of the
key loop once a key block is entirely in the future**, masking within the diagonal block, and carrying
online-softmax stats. (Block loops OK.)

In [ ]:
def flash_attention_causal(Q, K, V, block_size):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(3)
    T, d, dv = 9, 5, 4
    Q = rng.standard_normal((T, d)); K = rng.standard_normal((T, d)); V = rng.standard_normal((T, dv))
    S = np.where(np.tril(np.ones((T, T), bool)), Q @ K.T / np.sqrt(d), -np.inf)
    full = _softmax(S) @ V
    for bs in (1, 2, 4, T):
        assert np.allclose(flash_attention_causal(Q, K, V, bs), full, atol=1e-10), bs
    print("Q4 OK")

_q4()

## Q5 — Sliding-window attention via `sliding_window_view`  ·  `[hard · tensor trick]`

**Papers:** Longformer (Beltagy et al. 2020, arXiv:2004.05150); Mistral's sliding-window attention.
**Why it matters:** each query attends only to the last `w` keys, giving linear cost. **The trick:**
front-pad K/V by `w−1`, use `np.lib.stride_tricks.sliding_window_view` to extract each query's local
window as a `(T, w, ·)` tensor (a *view*, no `T×T` matrix), mask the padded slots, then `einsum`.

Implement `sliding_window_attention(Q, K, V, w) -> O (T,dv)`: query `t` attends to keys
`[t−w+1 … t]` (causal local window), softmax over the window. Must equal full attention with a banded
causal mask. No per-element loops.

In [ ]:
def sliding_window_attention(Q, K, V, w):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(4)
    T, d, dv, w = 8, 5, 4, 3
    Q = rng.standard_normal((T, d)); K = rng.standard_normal((T, d)); V = rng.standard_normal((T, dv))
    out = sliding_window_attention(Q, K, V, w)
    assert out.shape == (T, dv)
    idx = np.arange(T)
    band = (idx[:, None] >= idx[None, :]) & (idx[:, None] - idx[None, :] <= w - 1)
    full = _softmax(np.where(band, Q @ K.T / np.sqrt(d), -np.inf)) @ V
    assert np.allclose(out, full, atol=1e-10)
    causal = _softmax(np.where(np.tril(np.ones((T, T), bool)), Q @ K.T / np.sqrt(d), -np.inf)) @ V
    assert np.allclose(sliding_window_attention(Q, K, V, T), causal, atol=1e-10)   # w>=T -> full causal
    print("Q5 OK")

_q5()

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def _sm(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def ref_online_softmax_stats(x, block_size):
    x = np.asarray(x, float)
    m, l = -np.inf, 0.0
    for i in range(0, len(x), block_size):
        b = x[i:i + block_size]
        bm = b.max()
        nm = max(m, bm)
        l = l * np.exp(m - nm) + np.exp(b - nm).sum()
        m = nm
    return m, l


def ref_flash_attention(Q, K, V, block_size):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    Tq, d = Q.shape; Tk = K.shape[0]; dv = V.shape[1]
    scale = 1.0 / np.sqrt(d)
    m = np.full(Tq, -np.inf); l = np.zeros(Tq); O = np.zeros((Tq, dv))
    for j in range(0, Tk, block_size):
        Kb, Vb = K[j:j + block_size], V[j:j + block_size]
        S = Q @ Kb.T * scale                          # (Tq, bs)
        bm = S.max(axis=1)
        nm = np.maximum(m, bm)
        p = np.exp(S - nm[:, None])                    # (Tq, bs)
        alpha = np.exp(m - nm)                         # rescale running stats
        l = l * alpha + p.sum(axis=1)
        O = O * alpha[:, None] + p @ Vb
        m = nm
    return O / l[:, None]


def ref_attend_step(q_t, k_t, v_t, K_cache, V_cache):
    K = np.concatenate([K_cache, k_t[None, :]], axis=0)
    V = np.concatenate([V_cache, v_t[None, :]], axis=0)
    s = K @ q_t / np.sqrt(q_t.shape[0])
    p = _sm(s[None, :])[0]
    return p @ V, K, V


def ref_flash_attention_causal(Q, K, V, block_size):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    T, d = Q.shape; dv = V.shape[1]; scale = 1.0 / np.sqrt(d)
    O = np.zeros((T, dv))
    for i0 in range(0, T, block_size):
        ie = min(i0 + block_size, T)
        Qi = Q[i0:ie]; rows = np.arange(i0, ie)
        m = np.full(ie - i0, -np.inf); l = np.zeros(ie - i0); Oi = np.zeros((ie - i0, dv))
        for j0 in range(0, T, block_size):
            if j0 > ie - 1:                            # key block entirely in the future -> skip rest
                break
            je = min(j0 + block_size, T)
            Kb, Vb = K[j0:je], V[j0:je]; cols = np.arange(j0, je)
            S = Qi @ Kb.T * scale
            S = np.where(rows[:, None] >= cols[None, :], S, -np.inf)   # in-block causal mask
            bm = S.max(axis=1)
            nm = np.maximum(m, bm)
            p = np.exp(S - nm[:, None])
            alpha = np.exp(m - nm)
            l = l * alpha + p.sum(axis=1)
            Oi = Oi * alpha[:, None] + p @ Vb
            m = nm
        O[i0:ie] = Oi / l[:, None]
    return O


def ref_sliding_window_attention(Q, K, V, w):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    T, d = Q.shape; dv = V.shape[1]; pad = w - 1
    Kp = np.concatenate([np.zeros((pad, d)), K], axis=0)
    Vp = np.concatenate([np.zeros((pad, dv)), V], axis=0)
    Kw = np.lib.stride_tricks.sliding_window_view(Kp, w, axis=0).transpose(0, 2, 1)   # (T, w, d)
    Vw = np.lib.stride_tricks.sliding_window_view(Vp, w, axis=0).transpose(0, 2, 1)   # (T, w, dv)
    S = np.einsum("td,twd->tw", Q, Kw) / np.sqrt(d)
    p = np.arange(w)[None, :]; t = np.arange(T)[:, None]
    valid = (t - (w - 1) + p) >= 0                     # mask the front padding
    S = np.where(valid, S, -np.inf)
    A = _sm(S)
    return np.einsum("tw,twd->td", A, Vw)


_x = np.arange(10.0)
_m, _l = ref_online_softmax_stats(_x, 3)
assert np.isclose(_m, 9.0) and np.isclose(_l, np.exp(_x - 9).sum())
_Q = np.random.default_rng(9).standard_normal((4, 3)); _K = np.random.default_rng(8).standard_normal((6, 3)); _V = np.random.default_rng(7).standard_normal((6, 2))
assert np.allclose(ref_flash_attention(_Q, _K, _V, 4), _sm(_Q @ _K.T / np.sqrt(3)) @ _V)
assert ref_attend_step(np.zeros(3), np.zeros(3), np.ones(2), np.zeros((0, 3)), np.zeros((0, 2)))[1].shape == (1, 3)
print("reference solutions OK")